# PROCESO DE EXTRACCIÓN, TRANSFORMACIÓN Y CARGA DE DATOS SOBRE MUNICIPIOS Y PROVINCIAS ESPAÑOLAS (ETL)

## 0. Carga de librerías esenciales para el proceso

In [18]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
from ydata_profiling import ProfileReport

## 1. Extracción de datos a través de APIs oficiales y públicas

En primer lugar, ejecutamos el main.py de extracción de datos, para actualizar mantener los datos actualizados.

Nota: El alumno recomienda no actualizarlo si ya se poseen los datos, ya que puede suponer un tiempo de espera elevado

In [19]:
actualizar="n"
#actualizar = input("""¿Deseas actualizar los archivos de datos? (s/n)
 #        Ten en cuenta que puede tardar bastante en hacer las consultas, 
  #       por lo que si ya has cargado los datos una vez recomiendo no volver a hacerlo: """)

if actualizar=="s":
    %run Extractor/main.py
elif actualizar=="n":
    print("No se han actualizado los archivos de datos.")

No se han actualizado los archivos de datos.


En la carga inicial de datos, es importante recalcar que municipiosDF y provinciasDF son extraidos manualmente del Instituto Geográfico Nacional, que no posee una API abierta al público. Se comentará más en el apartado 2.1 de este Notebook

In [20]:
# Información de flujos de movimiento
INE_localidades = pd.read_csv('Extractor/data/processed/flujo_ine_localidad.csv', sep=',', encoding='UTF-8')
f_INE_provincias = pd.read_csv('Extractor/data/processed/flujo_ine_provincia.csv', sep=',', encoding='UTF-8')
INE_provincias = pd.read_csv('Extractor/data/processed/INE_provincias.csv', sep=',', encoding='UTF-8')
municipios_data = pd.read_csv('Extractor/data/processed/municipios_espana.csv', sep=',', encoding='UTF-8')

# Información económica hotelera
rentabilidad_H = pd.read_csv('Extractor/data/processed/rentabilidad_hotelera.csv', sep=",", encoding= 'UTF_8')

#Información demográfica y geográfica de provincias y municipios

municipiosDF = pd.read_csv('Extractor/data/raw/MUNICIPIOS.csv', sep=';', encoding='latin1')
provinciasDF = pd.read_csv('Extractor/data/raw/PROVINCIAS.csv', sep=';', encoding='latin1')

## 2. Transformación de la información extraida

### 2.1 Datos geográficos de municipios y provincias

In [21]:
# Primero revisamos la información y nombres de columnas en nuestros archivos descargados directamente del CNIG
provinciasDF.info()
print('-------------------------------------------------------')
municipiosDF.info()

municipiosDF.head(1).T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   COD_PROV            52 non-null     int64 
 1   PROVINCIA           52 non-null     object
 2   COD_CA              52 non-null     int64 
 3   COMUNIDAD_AUTONOMA  52 non-null     object
 4   CAPITAL             52 non-null     object
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
-------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8132 entries, 0 to 8131
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   COD_INE                   8132 non-null   int64 
 1   ID_REL                    8132 non-null   int64 
 2   COD_GEO                   8132 non-null   int64 
 3   COD_PROV                  8132 non-null   int64 
 4   PROVINCIA           

,0
COD_INE,1001000000
ID_REL,1010014
COD_GEO,1010
COD_PROV,1
PROVINCIA,Araba/Álava
NOMBRE_ACTUAL,Alegría-Dulantzi
POBLACION_MUNI,2961
SUPERFICIE,"1994,5872"
PERIMETRO,35069
COD_INE_CAPITAL,1001000101


Como la información de estos dos Dataframes está de por si bastante bien estructurada, no necesita ningún trabajo de transformación, por lo que procedemos a incluir en municipiosDF la información de su provincia para en futuros apartados crear ratios con esta información

In [22]:
#En primer lugar, vamos a combinar los dos DFs extraidos manualmente del centro de descargas del IGN 
# https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion

# Merge para unir información de provincia por su codigo
municipiosDF = municipiosDF.merge(provinciasDF[['COD_PROV','COD_CA', 'COMUNIDAD_AUTONOMA']], on='COD_PROV')

municipiosDF = municipiosDF.drop(['ID_REL', 'HOJA_MTN25', 'ORIGENCOOR', 'ORIGENALTITUD'], axis=1)

municipiosDF.head()


,COD_INE,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ALTITUD,COD_CA,COMUNIDAD_AUTONOMA
0,1001000000,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,"-2,512507724","42,84045247",568,16,País Vasco/Euskadi
1,1002000000,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,"-3,001015194","43,05265767",217,16,País Vasco/Euskadi
2,1003000000,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,"-2,564829379","43,05257873",325,16,País Vasco/Euskadi
3,1004000000,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,"-3,13052099","43,1217919",199,16,País Vasco/Euskadi
4,1006000000,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,"-2,872270813","42,72340924",466,16,País Vasco/Euskadi


### 2.2 Datos de caracter turístico extraidos mediante la API del Instituto Nacional de Estadística (INE)

Comenzamos con una revisión general de los datos. 
Como la API nos devuelve una estructura similar para todos estos Dataframes, el proceso va a ser similar, pero con casos especiales para cada situación.

In [23]:
#INE_localidades.head()
#INE_localidades.shape --> Resultado: (6610, 11)
INE_localidades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6610 entries, 0 to 6609
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   COD                     6610 non-null   object 
 1   Nombre                  6610 non-null   object 
 2   T3_Unidad               6610 non-null   object 
 3   T3_Escala               6610 non-null   object 
 4   Fecha                   6610 non-null   object 
 5   T3_Periodo              6610 non-null   object 
 6   T3_TipoDato             6610 non-null   object 
 7   Anyo                    6610 non-null   int64  
 8   Valor                   5560 non-null   float64
 9   MetaData_json           6610 non-null   object 
 10  tabla_id                6610 non-null   int64  
 11  _meta.fecha_extraccion  6610 non-null   object 
dtypes: float64(1), int64(2), object(9)
memory usage: 619.8+ KB


In [24]:
INE_provincias.head()
#INE_provincias[INE_provincias['INE_EOH_PROV.indicador']=='Viajero'].head()
#INE_provincias.info()

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,DPOP1,Total Nacional. Total. Total habitantes. Perso...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,47385107.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
1,DPOP2,Total Nacional. Hombres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,23222953.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
2,DPOP3,Total Nacional. Mujeres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,24162154.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
3,DPOP160,Albacete. Total. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,386464.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-07T22:13:34.997660
4,DPOP161,Albacete. Hombres. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,193205.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-07T22:13:34.997660


Cambios necesarios:
- INE_localidades y localidades_data se deben combinar
- En f_INE_provincias, INE_Provincias, Provincias_data y OpenStreetDF se deben combinar

El resultado debe ser dos DataFrames, uno para localidades, y otro para provincias. Posteriormente se deberá hacer un Left Join en las localidades para los municipios. Posteriormente, se deben combinar ambos Dfs

Comenzamos filtrando y reduciendo las columnas en localidades_data

In [25]:
municipios_data.nunique()

COD                       24396
Nombre                    24345
T3_Unidad                     1
T3_Escala                     1
Fecha                         1
T3_Periodo                    1
T3_TipoDato                   1
Anyo                          1
Valor                      6179
MetaData_json             24396
tabla_id                      1
_meta.fecha_extraccion        1
dtype: int64

In [26]:
# Primero, dividimos la columna de 'Nombre' usando el metodo .str.split()
columnas = ['Localidad', 'Genero', 'Metrica', 'Unidad']
municipios_data[columnas] = municipios_data['Nombre'].str.split('.', n=3, expand=True)

for col in columnas:
    municipios_data[col] = municipios_data[col].str.strip() #Eliminamos los espacios al inicio y final con un bucle

# Suprimimos las columnas que no vamos a necesitar
municipios_data.drop(columns=['Nombre', 'Fecha', 'T3_Unidad', 'T3_Escala', 'T3_Periodo', 'T3_TipoDato','Anyo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                     )

# Como nos interesa la poblacion total por municipio, filtramos Genero == 'Total
municipios_data = municipios_data[municipios_data['Genero'] == 'Total']

# Comprobamos el resultado
print(municipios_data.head())

print('--------------------------------------------------------------------------------------------------------')
print('Nota: realmente solo nos interesa COD, Valor y localidad, pero el resto de columnas aportan contexto')

          COD   Valor Localidad Genero           Metrica     Unidad
0   DPOP19723    73.0    Ababuj  Total  Total habitantes  Personas.
3   DPOP17671   849.0    Abades  Total  Total habitantes  Personas.
6    DPOP4663   337.0    Abadía  Total  Total habitantes  Personas.
9   DPOP12721  2239.0    Abadín  Total  Total habitantes  Personas.
12  DPOP22525  7768.0   Abadiño  Total  Total habitantes  Personas.
--------------------------------------------------------------------------------------------------------
Nota: realmente solo nos interesa COD, Valor y localidad, pero el resto de columnas aportan contexto


A continuación, adaptaremos INE_localidades.

En la columna T3_Unidad tenemos dos valores, Viajeros y pernoctaciones, por lo que nos interesa quedarnos con ambos pero en distintas columnas.

In [27]:
INE_localidades.head(2) #2 para que ocupe poco espacio en la salida

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-07-01T00:00:00.000+02:00,M07,Provisional,2026,18734.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-07T22:13:49.539679
1,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-06-01T00:00:00.000+02:00,M06,Provisional,2026,19464.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-07T22:13:49.539679


In [28]:
#Como el DF tiene dos niveles en el mismo origen de datos, vamos a partirlo en dos y despues unir sus columnas resultantes.

#El primer nivel será en INE_localidades2
INE_localidades2 = INE_localidades[(~INE_localidades['Nombre'].str.startswith('Nacional'))]
columnas = ['localidad', 'metrica', 'Campos', 'Origen', 'Tipo']

# Split de las columnas iniciales
INE_localidades2[columnas] = INE_localidades2['Nombre'].str.split('.', n=4, expand=True)

INE_localidades2.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion', 'Tipo'],
                     inplace=True
                     )

# Exploración de distintas métricas
print('Valores únicos en cada columna:')
print(INE_localidades2.nunique()) 
print('-----------------------------')
print(INE_localidades2.Anyo.value_counts())
print('-----------------------------')
print(INE_localidades2.metrica.value_counts())
print('-----------------------------')
print(INE_localidades2.Origen.value_counts())
print('-----------------------------')
# Conclusiones:
 # Decidimos quedarnos con el año 2026
 # Solo nos interesa mantener viajeros y pernoctaciones (Total categorias es el total)
 # Nos interesa diferenciar el origen en dos columnas, pero cambiando los nombres


INE_localidades2 = (INE_localidades2[(INE_localidades2['metrica'] != ' Total categorías')]
                    .rename(columns= {'T3_Periodo': 'periodo'}) # Renombramos la columna para su posterior uso
                    )

INE_localidades2['Origen'] = (INE_localidades2['Origen']
                              .str.strip() #Para limpiar el texto, ya que .replace solo no daba resultado
                              .replace({'Residentes en España': 'Nacional',
                                        'Residentes en el Extranjero': 'Extranjero'}))


INE_localidades2 =(INE_localidades2.sort_values('Valor', ascending=False)
 .pivot_table(index= 'localidad',
                             columns= ['metrica', 'Origen'],
                             values= ['Valor', 'periodo'],
                             aggfunc= {'Valor': 'sum', 'periodo': 'first'})
 .reset_index())

INE_localidades2.columns = INE_localidades2.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('Resultado del Dataframe')
INE_localidades2.head(3)


Valores únicos en cada columna:
COD            164
T3_Unidad        2
T3_Periodo      12
Anyo             2
Valor         1350
localidad       40
metrica          3
Campos           4
Origen           2
dtype: int64
-----------------------------
Anyo
2026    1142
2025     812
Name: count, dtype: int64
-----------------------------
metrica
Viajero             905
Pernoctaciones      905
Total categorías    144
Name: count, dtype: int64
-----------------------------
Origen
Residentes en España           978
Residentes en el Extranjero    976
Name: count, dtype: int64
-----------------------------
Resultado del Dataframe


,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,Alcalá de Henares,184738.0,303156.0,85821.0,187146.0,M10,M05,M10,M09
1,Alcúdia,3742634.0,220368.0,589171.0,46472.0,M08,M09,M08,M09
2,Almuñécar,333276.0,418600.0,52614.0,138623.0,M10,M08,M05,M08


In [29]:
#El segundo nivel mencionado anteriormente será en INE_localidades_nac, con los valores de residentes en españa y extranjeros
# Ahora, repetimos con la otra parte del dataframe
INE_localidades_nac = INE_localidades[INE_localidades['Nombre'].str.startswith('Nacional')]

# Misma operación con la otra metrica, pero diferencia en las columnas que desagregamos
columnas= ['tipo', 'metrica', 'localidad', 'tipo residente']

INE_localidades_nac[columnas] = INE_localidades_nac['Nombre'].str.split('.', n=3, expand=True)

INE_localidades_nac.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato','tipo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                        )

# Revisamos un poco los valores que nos encontramos
print(INE_localidades_nac.nunique()) 
print('-----------------------------')
print(INE_localidades_nac.metrica.value_counts()) #Otra vez, elegimos 2026
print('-----------------------------')
print(INE_localidades_nac['tipo residente'].value_counts()) #Genera un problema, ya que algunas filas no presentan bien si es español o extranjero

#Primero, solucionaremos el problema de tipo residente, apoyandonos en numpy

condiciones = [INE_localidades_nac['tipo residente'].str.contains('España', case=False, na = False),
               INE_localidades_nac['tipo residente'].str.contains('extranjero', case=False, na = False)
               ]
eleccion = ['Nacional', 'Extranjero']

INE_localidades_nac['tipo turista'] = np.select(condiciones, eleccion, default= ' ')

# Modificamos el DF
INE_localidades_nac = (INE_localidades_nac[(INE_localidades_nac['Anyo'] == 2026) & #Nos quedamos con 2026 porque tiene más metricas
                                           (INE_localidades_nac['metrica'] != ' Establecimientos hoteleros')
                                           ] 
                       .rename(columns={'T3_Periodo': 'periodo'}) #Aprovechamos para renombrar esta columna
                       )                 

# metodo rapido para modificar valor, ya que será necesario en el siguiente
INE_localidades_nac.loc[INE_localidades_nac['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'

# Para lograr tener el mes de mayor valor en la pivotacion, primero ordenaremos para poder mantener el mes con mayor numero de viajeros
INE_localidades_nac = (INE_localidades_nac
                       .sort_values('Valor', ascending=False) #Ordenamos
                       .pivot_table(index=['localidad'],
                                    columns= ['metrica', 'tipo turista'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'}) #Pivotamos
                       .reset_index()
                       )

# Eliminamos bandas y unificamos en el nombre de columna
INE_localidades_nac.columns = INE_localidades_nac.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('-----------------------------')
print('Resultado del Dataframe:')
INE_localidades_nac.head(3)


COD                388
T3_Unidad            2
T3_Periodo          12
Anyo                 2
Valor             4077
metrica              3
localidad           74
tipo residente      52
dtype: int64
-----------------------------
metrica
Viajeros                      1728
Pernoctaciones                1728
Establecimientos hoteleros    1200
Name: count, dtype: int64
-----------------------------
tipo residente
Residentes en España.                                             1728
Residentes en el extranjero.                                      1728
03063-Denia. Residentes en España.                                  24
03063-Denia. Residentes en el extranjero.                           24
04066-Níjar. Residentes en España.                                  24
04066-Níjar. Residentes en el extranjero.                           24
07014-Capdepera. Residentes en España.                              24
07014-Capdepera. Residentes en el extranjero.                       24
07051-Sant Llorenç de

,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,Adeje,6069924.0,302222.0,859460.0,76113.0,M07,M07,M03,M07
1,Albacete,32020.0,171794.0,16601.0,111280.0,M02,M05,M04,M05
2,Albarracín,6936.0,37844.0,4255.0,20366.0,M05,M04,M03,M04


Antes de concatenar (anexar) de vuelta los dos dataframes, debemos solucionar la diferencia entre las columnas resultado las operaciones paralelas

In [30]:
#Unificados nombres de columnas
columnas = (INE_localidades_nac.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )
# Aplicamos
INE_localidades2.columns = columnas
INE_localidades_nac.columns = columnas

# Unimos los Dfs
INE_localidades = pd.concat([INE_localidades2, INE_localidades_nac], ignore_index=True)

# Filtramos para suprimir que no tienen valores
INE_localidades = INE_localidades[INE_localidades[INE_localidades.columns[1]] > 0]

# Comprobamos si hay algun duplicado
print(INE_localidades.value_counts('LOCALIDAD').sort_values(ascending=False).head(5)) 

# Revisamos
INE_localidades.sort_values('LOCALIDAD').head()

LOCALIDAD
Adeje         1
Albacete      1
Albarracín    1
Algeciras     1
Alicante      1
Name: count, dtype: int64


,LOCALIDAD,VALOR PERNOCTACIONES EXTRANJERO,VALOR PERNOCTACIONES NACIONAL,VALOR VIAJERO EXTRANJERO,VALOR VIAJERO NACIONAL,PERIODO PERNOCTACIONES EXTRANJERO,PERIODO PERNOCTACIONES NACIONAL,PERIODO VIAJERO EXTRANJERO,PERIODO VIAJERO NACIONAL
38,Adeje,6069924.0,302222.0,859460.0,76113.0,M07,M07,M03,M07
39,Albacete,32020.0,171794.0,16601.0,111280.0,M02,M05,M04,M05
40,Albarracín,6936.0,37844.0,4255.0,20366.0,M05,M04,M03,M04
41,Algeciras,73041.0,88538.0,43367.0,43260.0,M07,M03,M03,M03
42,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06


Pasando a los datos agregados por provincias, el flujo de transformaciones será similar a los dfs de localidades, con pequeñas variaciones

In [31]:
# Comenzamos revisando los
print(f_INE_provincias.nunique())
print('-----------------------------')
print(f_INE_provincias['T3_Unidad'].value_counts())
print('-----------------------------')
print(f_INE_provincias.Nombre.value_counts())

#NOTA: en este DF están mezcladas provincias, comunidades y total nacional, conviene quedarse solo con provincias
# Para quedarnos con Provincias, se debe filtrar en la columna MetaData_json, que contiene el valor crudo en formato json, aunque no es necesario desagregarlo

f_INE_provincias = f_INE_provincias[f_INE_provincias['MetaData_json'].str.contains('Provincia', case=False, na=False)]

columnas=['Provincia', 'metrica', 'Origen']

f_INE_provincias[columnas] = f_INE_provincias['Nombre'].str.split('.', n=2, expand=True)
f_INE_provincias = f_INE_provincias.drop(columns= ['Nombre', 'T3_Escala', 'Fecha', 'T3_TipoDato', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion'])


#Filtramos años. En este caso nos quedamos con 2025 ya que contiene los doce meses
f_INE_provincias = (f_INE_provincias[f_INE_provincias['Anyo'] == 2025]
                     .rename(columns={'T3_Periodo': 'periodo'}))
                     
f_INE_provincias=f_INE_provincias.sort_values('Valor', ascending=False)


f_INE_provincias.loc[f_INE_provincias['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'
f_INE_provincias['metrica'] = f_INE_provincias['metrica'].str.strip()

f_INE_provincias = f_INE_provincias.pivot_table(index=['Provincia'],
                                    columns= ['metrica'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'})

f_INE_provincias.columns = f_INE_provincias.columns.map(' '.join)
f_INE_provincias.columns = (f_INE_provincias.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )

f_INE_provincias = f_INE_provincias.reset_index()
f_INE_provincias.head(2)

COD                        420
Nombre                     420
T3_Unidad                    2
T3_Escala                    1
Fecha                       12
T3_Periodo                  12
T3_TipoDato                  1
Anyo                         2
Valor                     4504
MetaData_json              420
tabla_id                     1
_meta.fecha_extraccion       1
dtype: int64
-----------------------------
T3_Unidad
Viajeros          2520
Pernoctaciones    2520
Name: count, dtype: int64
-----------------------------
Nombre
Nacional. Viajeros. Total categorías. Total.                          12
Nacional. Viajeros. Total categorías. Residentes en España.           12
Nacional. Viajeros. Total categorías. Residentes en el extranjero.    12
Nacional. Pernoctaciones. Total categorías. Total.                    12
Nacional. Pernoctaciones. Total categorías. Residentes en España.     12
                                                                      ..
Melilla. Viajeros. Residente

,Provincia,VALOR PERNOCTACIONES,VALOR VIAJERO,PERIODO PERNOCTACIONES,PERIODO VIAJERO
0,A Coruña,3811149.0,2142876.0,M08,M08
1,Alava,982555.0,456280.0,M08,M08


Para la rentabilidad hotelera, es importante conocer los dos indicadores que comparte este DataFrame
- **ADR (Average Daily Rate)**: Es el ingreso promedio por habitación ocupada (las habitaciones no alguiladas no cuentan)
- **RevPar (Revenue Per Available Room)**: Ingreso promedio por habitación disponible (ocupadas y disponibles).

In [32]:
#rentabilidad_H.info()

rentabilidad_H = rentabilidad_H.copy()

rentabilidad_H = rentabilidad_H[rentabilidad_H['MetaData_json'].str.contains('Provincias')]
columnas = ['Provincia', 'Metrica', 'Categoria', 'TipoDato']

rentabilidad_H[columnas] = rentabilidad_H['Nombre'].str.split('.', n=3, expand=True)

#Normalizamos nombres de columnas y posteriormente suprimimos las que no necesitamos
for col in columnas:
    rentabilidad_H[col] = (rentabilidad_H[col]
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace(r'\s+', ' ', regex= True) # suprimimos dobles espaciados
            .str.replace(r'[^\w\s]', '', regex= True)) # Truco para eliminar signos de puntuación

rentabilidad_H = rentabilidad_H.drop(['Nombre', 'T3_TipoDato', 'T3_Escala', 'T3_Unidad', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion', 'Categoria'], axis=1)


#Primero, revisamos los datos que nos podemos encontrar
print(rentabilidad_H.nunique())

# Renombramos para acortar
rentabilidad_H['Metrica'] = rentabilidad_H['Metrica'].replace({'Ingresos por habitación disponible RevPAR': 'RevPar',
                                            'Tarifa media diaria ADR':'ADR'})

# Reemplazamos para acortar TipoDato
rentabilidad_H['TipoDato'] =rentabilidad_H['TipoDato'].replace({'Tasa de variación interanual': 'Tasa Var'})

# Ordenamos  por valor mas algo y renombramos Periodo, para la posterior pivotacion
rentabilidad_H = (rentabilidad_H
         .sort_values('Valor', ascending= False)
         .rename(columns={'T3_Periodo':'Periodo'})
         )


# Sacamos la media de ADR por provincia como un dato extra.
# Es importante remarcar que ADR es ya de por si un ingreso medio de habitaciones ocupadas
# Por lo tanto, hacer la media por provincia no desvirtua tanto el dato como la media de RevPar 
rentAnual=(rentabilidad_H
           .groupby(['Provincia', 'TipoDato', 'Metrica'], as_index=False)['Valor'].mean())

# Filtramos los datos para quedarnos con el ADR medio de cada provincia
rentAnual = rentAnual[(rentAnual['Metrica'] == 'ADR') & (rentAnual['TipoDato'] == 'Dato') ]

# Pivotamos tablas, como hemos ordenado por valores mas alto podemos mantener Valor y periodo en first
rentabilidad_H = rentabilidad_H.pivot_table(index=['Provincia'],
                          columns=['Metrica','TipoDato'],
                          values=['Valor', 'Periodo'],
                          aggfunc={'Valor':'first', 'Periodo':'first'},
                          ) #Como ADR y RevPar son indicadores, no tiene sentido sumarlos

# Unimos las bandas y columnas para tener un unico nivel de columnas
rentabilidad_H.columns = rentabilidad_H.columns.map(' '.join)

# Reseteamos indices (Provincia)
rentabilidad_H = rentabilidad_H.reset_index()

rentabilidad_H.head()

COD            208
Fecha           12
T3_Periodo      12
Anyo             2
Valor         2182
Provincia       52
Metrica          2
TipoDato         2
dtype: int64


,Provincia,Periodo ADR Dato,Periodo ADR Tasa Var,Periodo RevPar Dato,Periodo RevPar Tasa Var,Valor ADR Dato,Valor ADR Tasa Var,Valor RevPar Dato,Valor RevPar Tasa Var
0,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79
1,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54
2,Almería,M08,M09,M08,M09,163.14,16.26,137.15,19.80
3,ArabaÁlava,M07,M06,M07,M10,117.12,13.91,89.31,27.88
4,Asturias,M08,M09,M08,M12,120.92,9.98,95.97,18.42


In [33]:
# Unimos las dos tablas trabjadas en la celda anterior
rentabilidad_H = rentabilidad_H.merge(rentAnual, how= 'left', on= 'Provincia', suffixes=('', '_A'))

# Borramos rentAnual para ahorrar memoria
del rentAnual

#rentabilidad_H = rentabilidad_H.drop(['Periodo RevPar Tasa Var' ], axis= 1 )
rentabilidad_H.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Provincia                52 non-null     object 
 1   Periodo ADR Dato         52 non-null     object 
 2   Periodo ADR Tasa Var     52 non-null     object 
 3   Periodo RevPar Dato      52 non-null     object 
 4   Periodo RevPar Tasa Var  52 non-null     object 
 5   Valor ADR Dato           52 non-null     float64
 6   Valor ADR Tasa Var       52 non-null     float64
 7   Valor RevPar Dato        52 non-null     float64
 8   Valor RevPar Tasa Var    52 non-null     float64
 9   TipoDato                 52 non-null     object 
 10  Metrica                  52 non-null     object 
 11  Valor                    52 non-null     float64
dtypes: float64(5), object(7)
memory usage: 5.0+ KB


Para finalizar esta parte del proyecto, guardamos los archivos ya trabajados en la carpeta Datos Procesados

In [35]:
# Finalmente, guardamos todos los DataFrames ya revisados y procesados

carpeta = Path('Datos procesados')
carpeta.mkdir(parents=True, exist_ok=True) # Por si no existe, se crea la carpeta



# Datos geográficos
municipiosDF.to_csv(carpeta / 'MunicipiosGeo.csv', index=False, sep=';', encoding='UTF-8')
provinciasDF.to_csv(carpeta / 'ProvinciasGeo.csv', index=False, sep=';', encoding='UTF-8')


municipios_data.to_csv(carpeta / 'MunicipiosData.csv', index=False, sep=';', encoding='UTF-8')
INE_provincias.to_csv(carpeta / 'INE_Provincias.csv', index=False, sep=';', encoding='UTF-8')
INE_localidades.to_csv(carpeta / 'INE_Municipios.csv', index=False, sep=';', encoding='UTF-8')
f_INE_provincias.to_csv(carpeta / 'INE_Provincias_Pernoctaciones.csv', index=False, sep=';', encoding='UTF-8')
rentabilidad_H.to_csv(carpeta / 'Provincias_Rentabilidad_Hotelera.csv', index=False, sep=';', encoding='UTF-8')
